# Hand Puzzle

Interactive hand-controlled jigsaw puzzle using OpenCV and MediaPipe.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import math
import random
import os
import urllib.request
import tkinter as tk
from tkinter import filedialog

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

MODEL_PATH = "hand_landmarker.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
BOARD_SIZE = 320
REFERENCE_ALPHA = 0.16
PINCH_THRESHOLD = 45
GRAB_DISTANCE = 65
SNAP_DISTANCE = 55
SMOOTHING = 0.38
TAB_RATIO = 0.28

def select_image():
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    path = filedialog.askopenfilename(
        title="Choose an image",
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp *.webp"), ("All files", "*.*")]
    )
    root.destroy()
    return path

def choose_difficulty():
    print("\n1 - Easy   3x3\n2 - Medium 4x4\n3 - Hard   5x5\n")
    while True:
        choice = input("Choose difficulty: ").strip()
        if choice == "1": return 3
        if choice == "2": return 4
        if choice == "3": return 5
        print("Enter 1, 2 or 3.")

def download_model():
    if not os.path.exists(MODEL_PATH):
        print("Downloading MediaPipe model...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
        print("Model downloaded.")

def create_detector():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=1,
        min_hand_detection_confidence=0.6,
        min_tracking_confidence=0.6
    )
    return vision.HandLandmarker.create_from_options(options)
def load_image(path):
    image = cv2.imread(path)

    if image is None:
        raise RuntimeError("Could not load image.")

    h, w = image.shape[:2]

    scale = min(
        BOARD_SIZE / w,
        BOARD_SIZE / h
    )

    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))

    image = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    canvas = np.zeros(
        (BOARD_SIZE, BOARD_SIZE, 3),
        dtype=np.uint8
    )

    x = (BOARD_SIZE - new_w) // 2
    y = (BOARD_SIZE - new_h) // 2

    canvas[
        y:y + new_h,
        x:x + new_w
    ] = image

    return canvas
    
def create_pieces(image, grid_size):
    pieces = []
    cell = BOARD_SIZE // grid_size
    for row in range(grid_size):
        for col in range(grid_size):
            x1, y1 = col * cell, row * cell
            x2 = BOARD_SIZE if col == grid_size - 1 else (col + 1) * cell
            y2 = BOARD_SIZE if row == grid_size - 1 else (row + 1) * cell
            crop = image[y1:y2, x1:x2].copy()
            pieces.append({"image": crop, "row": row, "col": col, "position": [0.0, 0.0], "locked": False})
    return pieces
     

def get_board_position(width, height):
    return ((width - BOARD_SIZE) // 2, (height - BOARD_SIZE) // 2)

def place_pieces(pieces, width, height, board_x, board_y):
    for index, piece in enumerate(pieces):
        h, w = piece["image"].shape[:2]
        if index % 2 == 0:
            min_x, max_x = 10, board_x - w - 15
        else:
            min_x, max_x = board_x + BOARD_SIZE + 15, width - w - 10
        min_y, max_y = 70, height - h - 70
        x = random.randint(int(min_x), int(max_x)) if max_x > min_x else 20
        y = random.randint(int(min_y), int(max_y)) if max_y > min_y else 100
        piece["position"] = [float(x), float(y)]
def draw_board(frame, image, board_x, board_y, grid_size):
    frame_h, frame_w = frame.shape[:2]

    x1 = max(0, int(board_x))
    y1 = max(0, int(board_y))
    x2 = min(frame_w, int(board_x + BOARD_SIZE))
    y2 = min(frame_h, int(board_y + BOARD_SIZE))

    if x2 <= x1 or y2 <= y1:
        return

    region = frame[y1:y2, x1:x2]

    target_w = x2 - x1
    target_h = y2 - y1

    resized_image = cv2.resize(
        image,
        (target_w, target_h),
        interpolation=cv2.INTER_AREA
    )

    faded = cv2.addWeighted(
        resized_image,
        REFERENCE_ALPHA,
        region,
        1.0 - REFERENCE_ALPHA,
        0
    )

    frame[y1:y2, x1:x2] = faded

    cell_w = target_w / grid_size
    cell_h = target_h / grid_size

    for i in range(grid_size + 1):
        x = int(x1 + i * cell_w)
        cv2.line(
            frame,
            (x, y1),
            (x, y2 - 1),
            (255, 255, 255),
            1
        )

    for i in range(grid_size + 1):
        y = int(y1 + i * cell_h)
        cv2.line(
            frame,
            (x1, y),
            (x2 - 1, y),
            (255, 255, 255),
            1
        )

    cv2.rectangle(
        frame,
        (x1, y1),
        (x2 - 1, y2 - 1),
        (255, 255, 255),
        2
    )

def draw_piece(frame, piece, selected=False):
    image = piece["image"]
    h, w = image.shape[:2]
    x, y = int(piece["position"][0]), int(piece["position"][1])
    fh, fw = frame.shape[:2]
    if x < 0 or y < 0 or x + w >= fw or y + h >= fh: return
    frame[y:y + h, x:x + w] = image
    border_color = (0, 255, 255) if selected else (255, 255, 255)
    if piece["locked"]: border_color = (0, 255, 0)
    cv2.rectangle(frame, (x, y), (x + w - 1, y + h - 1), border_color, 3)
    
def get_hand_points(hand, width, height):
    index, thumb = hand[8], hand[4]
    return ((int(index.x * width), int(index.y * height)), (int(thumb.x * width), int(thumb.y * height)))

def is_pinching(index, thumb):
    return math.hypot(index[0] - thumb[0], index[1] - thumb[1]) < PINCH_THRESHOLD

def get_piece_center(piece):
    h, w = piece["image"].shape[:2]
    return (piece["position"][0] + w / 2, piece["position"][1] + h / 2)

def get_target_center(board_x, board_y, grid_size, row, col):
    cell = BOARD_SIZE / grid_size
    return (int(board_x + col * cell + cell / 2), int(board_y + row * cell + cell / 2))

def draw_camera_circle(frame, camera):
    diameter, margin = 155, 25
    preview = cv2.resize(camera, (diameter, diameter))
    mask = np.zeros((diameter, diameter), dtype=np.uint8)
    center = (diameter // 2, diameter // 2)
    cv2.circle(mask, center, diameter // 2, 255, -1)
    x, y = frame.shape[1] - diameter - margin, frame.shape[0] - diameter - margin
    roi = frame[y:y + diameter, x:x + diameter]
    roi[mask == 255] = preview[mask == 255]
    cv2.circle(frame, (x + diameter // 2, y + diameter // 2), diameter // 2, (255, 255, 255), 3)

def main():
    image_path = select_image()
    if not image_path:
        print("No image selected.")
        return
    grid_size = choose_difficulty()
    download_model()
    detector = create_detector()
    puzzle_image = load_image(image_path)
    pieces = create_pieces(puzzle_image, grid_size)
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam.")
    ret, frame = cap.read()
    if not ret:
        cap.release()
        return
    height, width = frame.shape[:2]
    board_x, board_y = get_board_position(width, height)
    place_pieces(pieces, width, height, board_x, board_y)
    grabbed_index = None

    while True:
        ret, camera = cap.read()
        if not ret: break
        camera = cv2.flip(camera, 1)
        frame = camera.copy()
        height, width = frame.shape[:2]
        rgb = cv2.cvtColor(camera, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)
        hands = result.hand_landmarks if result.hand_landmarks else []
        finger = thumb = None
        if hands: finger, thumb = get_hand_points(hands[0], width, height)
        pinching = finger is not None and thumb is not None and is_pinching(finger, thumb)

        if grabbed_index is None and finger is not None and pinching:
            for i in range(len(pieces) - 1, -1, -1):
                piece = pieces[i]
                if piece["locked"]: continue
                center = get_piece_center(piece)
                distance = math.hypot(center[0] - finger[0], center[1] - finger[1])
                if distance < GRAB_DISTANCE:
                    selected = pieces.pop(i)
                    pieces.append(selected)
                    grabbed_index = len(pieces) - 1
                    break

        if grabbed_index is not None and finger is not None:
            piece = pieces[grabbed_index]
            ph, pw = piece["image"].shape[:2]
            target_x, target_y = finger[0] - pw / 2, finger[1] - ph / 2
            piece["position"][0] += (target_x - piece["position"][0]) * SMOOTHING
            piece["position"][1] += (target_y - piece["position"][1]) * SMOOTHING
            target = get_target_center(board_x, board_y, grid_size, piece["row"], piece["col"])
            center = get_piece_center(piece)
            distance = math.hypot(center[0] - target[0], center[1] - target[1])
            if distance < SNAP_DISTANCE:
                piece["position"] = [target[0] - piece["image"].shape[1] / 2, target[1] - piece["image"].shape[0] / 2]
                piece["locked"] = True
                grabbed_index = None

        if grabbed_index is not None and not pinching:
            grabbed_index = None

        draw_board(frame, puzzle_image, board_x, board_y, grid_size)

        for i, piece in enumerate(pieces):
            draw_piece(frame, piece, i == grabbed_index)

        if finger is not None:
            cv2.circle(frame, finger, 8, (0, 255, 255), -1)
            if pinching: cv2.circle(frame, finger, 16, (0, 255, 0), 2)

        solved = sum(1 for piece in pieces if piece["locked"])
        total = len(pieces)

        cv2.putText(frame, f"PUZZLE  {solved}/{total}", (25, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, "Pinch to grab", (25, height - 45), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
        cv2.putText(frame, "R = restart    Q = quit", (25, height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

        draw_camera_circle(frame, camera)

        if solved == total:
            cv2.rectangle(frame, (width // 2 - 220, 20), (width // 2 + 220, 80), (0, 0, 0), -1)
            cv2.putText(frame, "PUZZLE COMPLETE!", (width // 2 - 185, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 3)

        cv2.imshow("Hand Puzzle", frame)
        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"): break
        if key == ord("r"):
            pieces = create_pieces(puzzle_image, grid_size)
            place_pieces(pieces, width, height, board_x, board_y)
            grabbed_index = None

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()



1 - Easy   3x3
2 - Medium 4x4
3 - Hard   5x5



Choose difficulty:  2
